In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/playground-series-s6e10/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e10/train.csv
/kaggle/input/competitions/playground-series-s6e10/test.csv


In [2]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler, OneHotEncoder

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline 
from sklearn.impute import SimpleImputer

In [3]:
train = pd.read_csv("/kaggle/input/competitions/playground-series-s6e10/train.csv")
test = pd.read_csv("/kaggle/input/competitions/playground-series-s6e10/test.csv")
sample = pd.read_csv("/kaggle/input/competitions/playground-series-s6e10/sample_submission.csv")

train_cc = train.copy()
test_cc = test.copy()
sample_cc = sample.copy()

# Dataset Overview

In [4]:
print("Train dataset shape -- ", train.shape)
print("Test dataset shape -- ", test.shape)
print("Copied Train dataset shape -- ", train_cc.shape)
print("Copied Test dataset shape -- ", test_cc.shape)

Train dataset shape --  (699635, 23)
Test dataset shape --  (299844, 22)
Copied Train dataset shape --  (699635, 23)
Copied Test dataset shape --  (299844, 22)


In [5]:
train_cc.info()
print("="*50)
print("first 10 rows of the training dataset")
train_cc.head(10)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 699635 entries, 0 to 699634
Data columns (total 23 columns):
 #   Column                             Non-Null Count   Dtype  
---  ------                             --------------   -----  
 0   id                                 699635 non-null  int64  
 1   Age                                699635 non-null  int64  
 2   Flight Distance                    699635 non-null  int64  
 3   Inflight wifi service              699635 non-null  int64  
 4   Departure/Arrival time convenient  699635 non-null  int64  
 5   Ease of Online booking             699635 non-null  int64  
 6   Gate location                      699635 non-null  int64  
 7   Food and drink                     699635 non-null  int64  
 8   Online boarding                    699635 non-null  int64  
 9   Seat comfort                       699635 non-null  int64  
 10  Inflight entertainment             699635 non-null  int64  
 11  On-board service                   6996

,id,Age,Flight Distance,Inflight wifi service,Departure/Arrival time convenient,Ease of Online booking,Gate location,Food and drink,Online boarding,Seat comfort,...,Baggage handling,Checkin service,Cleanliness,Departure Delay in Minutes,Arrival Delay in Minutes,Gender,Customer Type,Type of Travel,Class,satisfaction
0,0,36,694,4,4,4,3,1,4,1,...,4,3,1,0,0.0,Female,Loyal Customer,Personal Travel,Eco,False
1,1,56,651,5,5,5,5,5,4,4,...,5,4,3,0,0.0,Male,Loyal Customer,Business travel,Business,True
2,2,31,1371,3,4,2,5,3,3,3,...,4,5,3,0,0.0,Female,Loyal Customer,Personal Travel,Eco,False
3,3,10,1616,3,5,3,3,3,3,3,...,3,4,3,0,0.0,Male,Loyal Customer,Personal Travel,Eco,False
4,4,23,481,3,4,3,4,5,3,5,...,5,3,5,5,0.0,Female,disloyal Customer,Business travel,Eco,False
5,5,34,584,4,4,4,4,1,4,1,...,4,4,1,0,0.0,Female,disloyal Customer,Business travel,Business,False
6,6,36,1763,5,5,5,5,3,4,4,...,4,3,3,0,0.0,Female,Loyal Customer,Business travel,Business,True
7,7,57,861,1,3,1,3,4,4,4,...,5,4,2,0,0.0,Female,Loyal Customer,Personal Travel,Eco,False
8,8,60,581,3,3,3,3,5,4,5,...,4,4,5,0,0.0,Female,Loyal Customer,Business travel,Business,True
9,9,44,588,3,3,2,2,2,3,2,...,4,1,2,0,0.0,Female,disloyal Customer,Business travel,Eco,False


In [6]:
# datatype identification

datatype_summary = pd.DataFrame({
    "Feature": train_cc.columns,
    "Data type": train_cc.dtypes.values,
    "category": [
        "Categorical" if train[col].dtype == "object" else "Numerical"
        for col in train_cc.columns
    ]
})

display(datatype_summary)

,Feature,Data type,category
0,id,int64,Numerical
1,Age,int64,Numerical
2,Flight Distance,int64,Numerical
3,Inflight wifi service,int64,Numerical
4,Departure/Arrival time convenient,int64,Numerical
5,Ease of Online booking,int64,Numerical
6,Gate location,int64,Numerical
7,Food and drink,int64,Numerical
8,Online boarding,int64,Numerical
9,Seat comfort,int64,Numerical


#### but upon looking very carefully we can see that columns "Departure/Arrival time convenient", "Ease of online booking", "Gate location", "food and drink", "online boarding", "seat comfort", "inflight entertainment", "on-board service", "Leg room service", "Baggage handling", "Checkin service", "Cleanliness" are all ordinal features. 



In [7]:
# given target is satisfaction as target
target = "satisfaction"

identifier_cols = [
    "id"
]

numerical_cols = [
    "Age",
    "Flight Distance",
    "Departure Delay in Minutes",
    "Arrival Delay in Minutes"
]

ordinal_cols = [
    "Inflight wifi service",
    "Departure/Arrival time convenient",
    "Ease of Online booking",
    "Gate location",
    "Food and drink",
    "Online boarding",
    "Seat comfort",
    "Inflight entertainment",
    "On-board service",
    "Leg room service",
    "Baggage handling",
    "Checkin service",
    "Cleanliness"
]

categorical_cols = [
    "Gender",
    "Customer Type",
    "Type of Travel",
    "Class"
]

print("Categorical Features:", categorical_cols)
print(f"Total Categorical Features: {len(categorical_cols)}")

print("\nOrdinal Features:", ordinal_cols)
print(f"Total Ordinal Features: {len(ordinal_cols)}")

print("\nNumerical Features:", numerical_cols)
print(f"Total Numerical Features: {len(numerical_cols)}")

print("\nIdentifier Features:", identifier_cols)
print(f"Total Identifier Features: {len(identifier_cols)}")

print("\nTarget Variable:", target)

Categorical Features: ['Gender', 'Customer Type', 'Type of Travel', 'Class']
Total Categorical Features: 4

Ordinal Features: ['Inflight wifi service', 'Departure/Arrival time convenient', 'Ease of Online booking', 'Gate location', 'Food and drink', 'Online boarding', 'Seat comfort', 'Inflight entertainment', 'On-board service', 'Leg room service', 'Baggage handling', 'Checkin service', 'Cleanliness']
Total Ordinal Features: 13

Numerical Features: ['Age', 'Flight Distance', 'Departure Delay in Minutes', 'Arrival Delay in Minutes']
Total Numerical Features: 4

Identifier Features: ['id']
Total Identifier Features: 1

Target Variable: satisfaction


In [8]:
for col in categorical_cols:
    unique_values = train_cc[col].dropna().unique()
    print(f"{col} -- {unique_values}")
    
print("="*50)

for col in numerical_cols:
    unique_values = train_cc[col].dropna().unique()
    print(f"{col} -- {unique_values}")

for col in ordinal_cols:
    unique_values = train_cc[col].dropna().unique()
    print(f"{col} -- {unique_values}")    

Gender -- ['Female' 'Male']
Customer Type -- ['Loyal Customer' 'disloyal Customer']
Type of Travel -- ['Personal Travel' 'Business travel']
Class -- ['Eco' 'Business' 'Eco Plus']
Age -- [36 56 31 10 23 34 57 60 44 32 28 26 24 48 49 41 45 22 42  9 30 55 29 20
 14 58 51 16 27 54 39 65 37 59 17 53 50 15 43 40 35 46 33 62 52 38 21 18
  8 47 61 25 70 63 19 69 12  7 68 11 13 67 64 66 72 77 71 78 73 74 75 80
 76 79 85]
Flight Distance -- [ 694  651 1371 ... 3002 1154 2048]
Departure Delay in Minutes -- [  0   5  25  24   3   6   2   1  17  10   4  12  47  42  18   8   7  15
  21   9  26  35  52  19  31  22  13 152  29  36  16  14  37  41  28  50
  33  11  34  40  45  23 153  20  27  38  44  43  39  30 205  55  32  61
  46  56  54  48  49 132 148  63 157  81 160 396  53  62  57  60 200 130
  51 151 158  67 150 145  66 140 185  83 136  58  59 168  64  71 156  70
 159 182 112 196 121 202 209 144 164 167 178 138 194  72 204  65 165 154
 131 186 183 137 135 193 139 149 238  68  73 170 176 173 198 

now for the ordinal features we can see 6 unique values which are 
0 -- not applicable

1 -- terrible or worse

2 -- poor or bad

3 -- average or meh

4 -- good or thumps up

5 -- excellent or noice

so this features can be sorted by ascending and descending order.

In [9]:
#Descriptive statistics of numerical features
numerical_features = ["Age", "Flight Distance", "Departure Delay in Minutes","Arrival Delay in Minutes"] #ring-number is excluded as it's a binary identifier

numerical_statistics = pd.DataFrame({
    "Minimum": train_cc[numerical_features].min(),
    "Maximum": train_cc[numerical_features].max(),
    "Mean": train_cc[numerical_features].mean(),
    "Median": train_cc[numerical_features].median()
})

display(numerical_statistics)

,Minimum,Maximum,Mean,Median
Age,7.0,85.0,39.001928,40.0
Flight Distance,67.0,4983.0,1353.778702,954.0
Departure Delay in Minutes,0.0,489.0,1.176209,0.0
Arrival Delay in Minutes,0.0,491.0,1.134280,0.0


In [10]:
#detailed descriptive statistics 

display(train_cc[numerical_features].describe().T)

,count,mean,std,min,25%,50%,75%,max
Age,699635.0,39.001928,13.800943,7.0,27.0,40.0,50.0,85.0
Flight Distance,699635.0,1353.778702,954.422566,67.0,631.0,954.0,1814.0,4983.0
Departure Delay in Minutes,699635.0,1.176209,5.982187,0.0,0.0,0.0,0.0,489.0
Arrival Delay in Minutes,699343.0,1.134280,5.749304,0.0,0.0,0.0,0.0,491.0


# Missing value analysis

In [11]:
#missing value
missing_summary = pd.DataFrame({
    "Missing Count": train_cc.isnull().sum(),
    "Missing Percentage": train_cc.isnull().mean() * 100
})

missing_summary = missing_summary[
    missing_summary["Missing Count"] > 0
].sort_values(
    by = "Missing Count",
    ascending = False
)

display(missing_summary)

,Missing Count,Missing Percentage
Arrival Delay in Minutes,292,0.041736


the main idea is to impute it by using median.

In [12]:
train_cc["id"].duplicated().sum()

np.int64(0)

In [13]:
#duplicate analysis

feature_cols = [
    col for col in train_cc.columns
    if col not in ["id", "satisfaction"]
]

duplicate_feature_rows = train_cc.duplicated(
    subset=feature_cols
).sum()

print("Duplicate feature combinations:", duplicate_feature_rows)

Duplicate feature combinations: 0
